# Germana: preparación y embeddings con MiniLM

Corpus: *Germana*, Edmond About, traducción de Tomás Orts-Ramos. Fuente: https://www.gutenberg.org/ebooks/29640 (archivo `germana.txt`).

Se seleccionó `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` ya que admite español, está preparado para similitud y agrupamiento de textos y produce vectores compactos de 384 dimensiones. Su configuración de Sentence Transformers limita la entrada a 128 tokens.

Se conserva un párrafo como documento. Para párrafos largos, se usaron fragmentos contiguos sin solapamiento y un promedio de sus embeddings ponderado por tokens de contenido. Esta solución evita truncar contenido, aunque puede diluir cambios de tema dentro de un párrafo. No se ajustan los pesos del modelo. Las stopwords se aplicarán posteriormente en CountVectorizer; se conservan aquí para mantener el contexto.

Este notebook cubre la preparación y los embeddings. El agrupamiento y la comparación de configuraciones de BERTopic quedan para el siguiente paso.


In [1]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import re
from datetime import datetime, timezone

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from huggingface_hub import HfApi
from sentence_transformers import SentenceTransformer

BASE = Path.cwd() if (Path.cwd() / "germana.txt").exists() else Path.cwd() / "BERTopic"
OUT = BASE / "resultados"
OUT.mkdir(exist_ok=True)
MODEL_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"


In [2]:
raw = (BASE / "germana.txt").read_text(encoding="utf-8-sig")
start = "*** START OF THE PROJECT GUTENBERG EBOOK GERMANA ***"
end = "*** END OF THE PROJECT GUTENBERG EBOOK GERMANA ***"
assert start in raw and end in raw, "Revisar los marcadores de Gutenberg"
body = raw.split(start, 1)[1].split(end, 1)[0]
# Inicio específico de esta edición: elimina portada e índice.
body = body[body.index("Hacia la mitad de la calle"):].split("\nNOTAS:", 1)[0]
blocks = [" ".join(p.split()) for p in re.split(r"\n\s*\n", body) if p.strip()]
rows, chapter, skip_title = [], 1, False
for block in blocks:
    if re.fullmatch(r"[IVXLCDM]+", block):
        chapter += 1
        skip_title = True
        continue
    if skip_title:
        skip_title = False
        continue
    if block == "FIN":
        continue
    # Retira llamadas de notas y las marcas de cursiva de la edición.
    text = re.sub(r"\[[A-F]\]", "", block).replace("_", "")
    rows.append({"parrafo_origen": len(rows) + 1, "capitulo": chapter,
                 "texto": text, "palabras": len(text.split())})
assert chapter == 15
original = pd.DataFrame(rows)
short_count = int((original.palabras < 5).sum())
df = original.loc[original.palabras >= 5].copy()
duplicates = int(df.duplicated("texto").sum())
df = df.drop_duplicates("texto").reset_index(drop=True)
df.insert(0, "segmento_id", [f"germana_{i:04d}" for i in range(1, len(df) + 1)])
assert len(df) >= 500
print({"parrafos_narrativos": len(original), "cortos_eliminados": short_count,
       "duplicados_eliminados": duplicates, "segmentos": len(df),
       "palabras_promedio": round(df.palabras.mean(), 2)})


{'parrafos_narrativos': 1316, 'cortos_eliminados': 139, 'duplicados_eliminados': 0, 'segmentos': 1177, 'palabras_promedio': np.float64(57.44)}


## Descargar una revisión identificable y medir tokens

La primera ejecución consulta la revisión del modelo; las siguientes reutilizan el identificador guardado en `resultados/modelo_revision.json`. Los archivos descargados se guardan en la caché local del proyecto.


In [3]:
revision_file = OUT / "modelo_revision.json"
if revision_file.exists():
    model_meta = json.loads(revision_file.read_text())
    assert model_meta["modelo"] == MODEL_ID
else:
    model_meta = {"modelo": MODEL_ID, "revision": HfApi().model_info(MODEL_ID).sha}
    revision_file.write_text(json.dumps(model_meta, indent=2))
model = SentenceTransformer(MODEL_ID, revision=model_meta["revision"],
                            cache_folder=str(BASE / ".cache"), device="cpu")
tokenizer = model.tokenizer
limit = model.max_seq_length
special_tokens = tokenizer.num_special_tokens_to_add(pair=False)
budget = limit - special_tokens
assert budget > 0
encoded = tokenizer(df.texto.tolist(), add_special_tokens=False, truncation=False,
                    return_attention_mask=False, return_token_type_ids=False,
                    verbose=False)["input_ids"]
df["tokens_con_especiales"] = [len(ids) + special_tokens for ids in encoded]
df["excede_limite"] = df.tokens_con_especiales > limit
print(f"Límite efectivo: {limit} tokens; presupuesto de contenido: {budget}")
print(f"Párrafos que lo exceden: {df.excede_limite.sum()} / {len(df)} "
      f"({df.excede_limite.mean():.2%})")
print(df[["palabras", "tokens_con_especiales"]].describe().round(2).to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(df.palabras, bins=35, color="#215c92", edgecolor="white")
axes[0].set(xlabel="Palabras por párrafo", ylabel="Segmentos", title="Longitud en palabras")
axes[1].hist(df.tokens_con_especiales, bins=35, color="#218b78", edgecolor="white")
axes[1].axvline(limit, color="#b83838", linestyle="--", label=f"Límite: {limit}")
axes[1].set(xlabel="Tokens por párrafo (incluye especiales)", ylabel="Segmentos", title="Longitud con MiniLM")
axes[1].legend()
fig.tight_layout()
fig.savefig(OUT / "histograma_longitudes.png", dpi=160)
plt.close(fig)


Límite efectivo: 128 tokens; presupuesto de contenido: 126
Párrafos que lo exceden: 256 / 1177 (21.75%)
       palabras  tokens_con_especiales
count   1177.00                1177.00
mean      57.44                  87.34
std       68.23                  97.66
min        5.00                   9.00
25%       13.00                  23.00
50%       28.00                  45.00
75%       78.00                 115.00
max      568.00                 787.00


## Embeddings sin pérdida por truncamiento

Se pasan los IDs originales de cada fragmento al modelo para evitar cambios al decodificar y volver a tokenizar. Cada fragmento recibe los tokens especiales del modelo y respeta su límite. Se promedian los vectores sin normalizar de los fragmentos, ponderando por cantidad de tokens de contenido; después se normaliza cada vector de párrafo a longitud uno. Los fragmentos no se convierten en documentos nuevos para BERTopic.


In [4]:
import torch

# Este tokenizador usa <s> y </s> en una secuencia individual.
assert tokenizer("", add_special_tokens=True)["input_ids"] == [tokenizer.cls_token_id, tokenizer.sep_token_id]
probe = tokenizer("Texto de prueba", add_special_tokens=False)["input_ids"]
assert tokenizer("Texto de prueba")["input_ids"] == [tokenizer.cls_token_id] + probe + [tokenizer.sep_token_id]
features, owners, weights = [], [], []
for paragraph_idx, ids in enumerate(encoded):
    assert ids
    for start_idx in range(0, len(ids), budget):
        chunk = ids[start_idx:start_idx + budget]
        input_ids = [tokenizer.cls_token_id] + chunk + [tokenizer.sep_token_id]
        prepared = {"input_ids": input_ids, "attention_mask": [1] * len(input_ids)}
        assert len(prepared["input_ids"]) <= limit
        features.append(prepared)
        owners.append(paragraph_idx)
        weights.append(len(chunk))

model.eval()
chunk_vectors = []
with torch.inference_mode():
    for first in range(0, len(features), 32):
        batch = tokenizer.pad(features[first:first + 32], padding=True, return_tensors="pt")
        output = model(dict(batch))
        chunk_vectors.append(output["sentence_embedding"].cpu().numpy())
chunk_vectors = np.concatenate(chunk_vectors)
embeddings = np.zeros((len(df), chunk_vectors.shape[1]), dtype=np.float32)
weight_sums = np.zeros(len(df), dtype=np.float32)
np.add.at(embeddings, owners, chunk_vectors * np.asarray(weights)[:, None])
np.add.at(weight_sums, owners, weights)
embeddings /= weight_sums[:, None]
embeddings /= np.maximum(np.linalg.norm(embeddings, axis=1, keepdims=True), 1e-12)
assert embeddings.shape == (len(df), 384)
assert np.isfinite(embeddings).all()
assert np.allclose(np.linalg.norm(embeddings, axis=1), 1, atol=1e-5)
df["fragmentos_embedding"] = np.bincount(owners, minlength=len(df))
np.save(OUT / "embeddings_minilm.npy", embeddings)
df.to_csv(OUT / "segmentos.csv", index=False)

summary = {**model_meta, "fecha_utc": datetime.now(timezone.utc).isoformat(),
    "corpus_sha256": hashlib.sha256(raw.encode()).hexdigest(),
    "segmentos": len(df), "cortos_eliminados": short_count,
    "duplicados_eliminados": duplicates,
    "palabras_total": int(df.palabras.sum()), "palabras_promedio": float(df.palabras.mean()),
    "palabras_mediana": float(df.palabras.median()),
    "limite_tokens": limit, "parrafos_excedidos": int(df.excede_limite.sum()),
    "porcentaje_excedido": float(100 * df.excede_limite.mean()),
    "fragmentos": len(features), "dimension_embedding": embeddings.shape[1],
    "estrategia": "fragmentos sin solapamiento, promedio ponderado por tokens, normalización L2 final",
    "versiones": {p: importlib.metadata.version(p) for p in
        ["sentence-transformers", "transformers", "torch", "numpy", "pandas", "matplotlib"]}}
(OUT / "resumen_preparacion.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2))
print(json.dumps(summary, ensure_ascii=False, indent=2))


{
  "modelo": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
  "revision": "e8f8c211226b894fcb81acc59f3b34ba3efd5f42",
  "fecha_utc": "2026-10-25T22:49:45.121862+00:00",
  "corpus_sha256": "0b217a737c5d5081d0bc0486d1380e253baa71169489671a809432a2f38107a3",
  "segmentos": 1177,
  "cortos_eliminados": 139,
  "duplicados_eliminados": 0,
  "palabras_total": 67601,
  "palabras_promedio": 57.43500424808836,
  "palabras_mediana": 28.0,
  "limite_tokens": 128,
  "parrafos_excedidos": 256,
  "porcentaje_excedido": 21.75021240441801,
  "fragmentos": 1555,
  "dimension_embedding": 384,
  "estrategia": "fragmentos sin solapamiento, promedio ponderado por tokens, normalización L2 final",
  "versiones": {
    "sentence-transformers": "5.1.2",
    "transformers": "4.57.6",
    "torch": "2.8.0",
    "numpy": "2.0.2",
    "pandas": "2.3.3",
    "matplotlib": "3.9.4"
  }
}


## Fuentes y uso de IA

- Corpus: https://www.gutenberg.org/ebooks/29640
- Modelo y configuración: https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
- Sentence-BERT: https://aclanthology.org/D19-1410/
- Presentación de clase: *BERT y BERTopic*, Dr. Rubén Alvarez, diapositivas 45–47.
- Se utilizó Codex (OpenAI), el 25 de octubre de 2026, para preparar el código y discutir la elección del modelo y el tratamiento de párrafos largos. Deben revisarse y explicarse estas decisiones en el reporte final.
